In [ ]:
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image

from src.data.data_loader import (val_transform, BINARY_CLASSES, TUMOR_CLASSES,
                                   SEED, CONFIDENCE_THRESHOLD)
from src.model.model import SimpleCNN, build_model_b, GradCAM
from src.utils.helpers import show_gradcam_grid

# ─── Config ─────────────────────────────────────────────
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_DIR    = Path('models')
MODEL_B_PATH = MODEL_DIR / 'model_b_3class.pth'
CONFIDENCE_THRESHOLD = 0.85

# ─── Charger Model A scratch ─────────────────────────────
params_model_a = {
    "shape_in":        (3, 224, 224),
    "initial_filters": 8,
    "num_fc1":         100,
    "dropout_rate":    0.5,
    "num_classes":     2
}
model_scratch_a = SimpleCNN(params_model_a)
model_scratch_a.load_state_dict(torch.load("weights_scratch_a.pt", map_location=DEVICE))
model_scratch_a = model_scratch_a.to(DEVICE).eval()

# ─── Charger Model B ─────────────────────────────────────
model_b = build_model_b()
model_b.load_state_dict(torch.load(MODEL_B_PATH, map_location=DEVICE))
model_b = model_b.to(DEVICE).eval()

print('Modèles chargés ')
print(f'Device : {DEVICE}')

In [ ]:
def predict_image(image_path, threshold=0.4):
    image  = Image.open(image_path).convert('RGB')
    tensor = val_transform(image).unsqueeze(0).to(DEVICE)

    # ── Stage 1 : Model A ────────────────────────────────
    with torch.no_grad():
        probs_a      = torch.exp(model_scratch_a(tensor))[0]
    conf_notumor = probs_a[0].item()
    conf_tumor   = probs_a[1].item()
    max_conf_a   = max(conf_notumor, conf_tumor)

    # Low confidence
    if max_conf_a < CONFIDENCE_THRESHOLD:
        return {
            'status'      : 'low_confidence',
            'tumor_type'  : None,
            'confidence_a': max_conf_a,
            'confidence_b': None,
            'message'     : f'Low confidence ({max_conf_a:.2%}): voir un radiologue'
        }

    pred_a = 1 if conf_tumor > threshold else 0

    # No tumor
    if pred_a == 0:
        return {
            'status'      : 'no_tumor',
            'tumor_type'  : None,
            'confidence_a': conf_notumor,
            'confidence_b': None,
            'message'     : f'Pas de tumeur détectée (confiance: {conf_notumor:.2%})'
        }

    # ── Stage 2 : Model B ────────────────────────────────
    with torch.no_grad():
        logits_b = model_b(tensor.to(DEVICE))
        probs_b  = F.softmax(logits_b, dim=1)[0]
    pred_b     = int(probs_b.argmax())
    conf_b     = probs_b[pred_b].item()
    tumor_type = TUMOR_CLASSES[pred_b]

    return {
        'status'      : 'tumor',
        'tumor_type'  : tumor_type,
        'confidence_a': conf_tumor,
        'confidence_b': conf_b,
        'message'     : (
            f'Tumeur détectée : {tumor_type.capitalize()} '
            f'(détection: {conf_tumor:.2%}, classification: {conf_b:.2%})'
        )
    }

print('Pipeline prêt ')

In [ ]:
from src.data.data_loader import TEST_ROOT

for class_folder in TEST_ROOT.iterdir():
    if not class_folder.is_dir():
        continue
    imgs = list(class_folder.glob('*'))[:1]
    for img_path in imgs:
        result     = predict_image(img_path)
        true_label = class_folder.name
        print(f'Vraie classe : {true_label}')
        print(f'Prédiction   : {result["message"]}')
        print()

In [ ]:
from src.data.data_loader import get_loaders

_, _, test_loader_a = get_loaders('binary')
_, _, test_loader_b = get_loaders('3class')

# Model A GradCAM
show_gradcam_grid(
    model_scratch_a, model_scratch_a.conv4,
    test_loader_a, BINARY_CLASSES,
    'Model A Grad-CAM',
    device    = DEVICE,
    GradCAM   = GradCAM,
    is_binary = True,
    threshold = 0.4
)

# Model B GradCAM
show_gradcam_grid(
    model_b, model_b.layer4[-1],
    test_loader_b, TUMOR_CLASSES,
    'Model B Grad-CAM',
    device    = DEVICE,
    GradCAM   = GradCAM,
    is_binary = False
)